In [1]:
pwd

'D:\\樓上志偉\\2025實測\\notebooks'

In [2]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
import seaborn as sns
import pickle

pd.options.mode.copy_on_write = True
fm.fontManager.addfont('..\\TaipeiSansTCBeta-Regular.ttf')
plt.rcParams["font.size"] = 14
plt.rcParams['font.family'] = 'Taipei Sans TC Beta'

In [3]:
# with open('..\\models\\model_6_spacing_8x8_HALF_LOAD.pkl', 'rb') as ff:
with open('..\\models\\model_6_spacing_8x8_FULL_LOAD.pkl', 'rb') as ff:
    model = pickle.load(ff)

In [4]:
print(f"係數: {model.coef_}")
print(f"截距: {model.intercept_}")

係數: [-0.07565586 -0.07036326]
截距: 385.83641632000825


In [5]:
# testing_load = "HALF_LOAD"
testing_load = "FULL_LOAD"
df_p = pd.read_csv(f'..\\data\\raw\\偉仁26噸\\B114AS0005_0327_FUEL_CONSUMPTION_{testing_load}_1ST.csv',
                 header=2,
                 usecols=['GPS_VehicleSpeed', 'OBD_EngineSpeed', 'OBD_EngineFuelRate'],
                )
df_p.columns = ['VehicleSpeed[km/h]', 'EngineSpeed[rpm]', 'EngineFuelRate[L/h]']

In [6]:
# 檔位判斷
# 忽略空檔、換檔間的操作情況
def v_to_n(v, g):
    """ 從車速[km/h]、檔位g，計算引擎轉速[rpm] """
    if gear_ratio := i_m_1_16.get(g):
        return (v*1000./60.) / (wheel_D*np.pi) * i_f * gear_ratio
    else:
        print("Gear Position Error!")

def predit_gear_position(v, n):
    """ 從車速[km/h]，計算1~16檔的可能引擎轉速，並取出最接近引擎轉速的檔位 """
    l=[]
    for g in range(1, 17):
        l.append(abs(v_to_n(v,g)-n))
    return l.index(min(l))+1

def engine_torque(R, i_m): # 此R的單位為[N]
    if R>=0: # 原稿為R>0
        return r / eff_m / eff_f / i_m / i_f * R
    else:
        return r * eff_m * eff_f / i_m / i_f * R

In [7]:
density_deisel = 836. # 柴油密度[kg/m3] @ VECTO

B = 2.6 # 車寬[m]
H = 3.75 # 車高[m]

# 變速箱齒比
i_m_1_16 = {1:14.68, 2:12.05, 3:9.92, 4:8.14, 5:6.78, 6:5.56, 7:4.57, 8:3.75, 9:3.22, 10:2.64, 11:2.17, 12:1.78, 13:1.49, 14:1.22, 15:1, 16:0.82}
i_f = 3.727 # 差速器減速比
eff_m = 0.98 # 傳動效率
eff_f = 0.95

# driven wheels: 318/80 R22.5
wheel_D = (318*0.80*2 + 22.5*25.4) / 1000. # [m]
r = wheel_D / 2 # [m]

W0 = 14975 # 空車重[kg]

In [8]:
df_p['grad[%]'] = 0
df_p['time[s]'] = [i for i in range(1, len(df_p)+1)]
df_p['acc[m/s^2]'] = (df_p.loc[0, 'VehicleSpeed[km/h]'] - 0.) / 3.6
for i in range(1, len(df_p)):
    df_p.loc[i, 'acc[m/s^2]'] = (df_p.loc[i, 'VehicleSpeed[km/h]'] - df_p.loc[i-1, 'VehicleSpeed[km/h]']) / 3.6

# 全載(90%)車重: 14975+(26000-14975)*0.9=24897.5, 半載(55%)車重: 14975+(26000-14975)*0.55=21038.75
if testing_load == "HALF_LOAD":
    df_p['W[kg]'] = 14975+(26000-14975)*0.55
else:
    df_p['W[kg]'] = 14975+(26000-14975)*0.9

df_p['predit_gear_position'] = df_p.apply(lambda x: predit_gear_position(x['VehicleSpeed[km/h]'], x['EngineSpeed[rpm]']), axis=1)
df_p['i_m'] = df_p['predit_gear_position'].apply(lambda x :i_m_1_16.get(x))
df_p['mu_r[kg/kg]'] = 0.00513 + 17.6 / df_p['W[kg]']
df_p['mu_aA[kg/(km/h)2]'] = 0.00299 * B * H - 0.000832
df_p['W_eq[kg]'] = (0.07 + 0.03*df_p['i_m']*df_p['i_m']) * W0
df_p['F_rr[N]'] = df_p['mu_r[kg/kg]'] * df_p['W[kg]'] * 9.81 # 調整單位為N
df_p['F_slope[N]'] = df_p['W[kg]'] * np.sin(np.arctan(df_p['grad[%]'] / 100)) * 9.81
df_p['F_air[N]'] = df_p['mu_aA[kg/(km/h)2]'] * df_p['VehicleSpeed[km/h]'] * df_p['VehicleSpeed[km/h]'] * 9.81
df_p['F_acc[N]'] = (df_p['W[kg]'] + df_p['W_eq[kg]']) * df_p['acc[m/s^2]']
df_p['R[N]'] = df_p['F_rr[N]'] + df_p['F_slope[N]'] + df_p['F_air[N]'] +df_p['F_acc[N]']
df_p['engine_torque[Nm]'] = df_p.apply(lambda x: engine_torque(x['R[N]'], x['i_m']), axis=1)
df_p['engine_power[kW]'] = df_p['engine_torque[Nm]'] * (df_p['EngineSpeed[rpm]'] * 2. * np.pi / 60.) / 1000.

df_p['predict_BSFC[g/kWh]'] = model.predict(df_p[['EngineSpeed[rpm]', 'engine_torque[Nm]']])
df_p['predict_FuelRate[L/h]'] = df_p['predict_BSFC[g/kWh]'] * df_p['engine_power[kW]'] / 1000. / density_deisel * 1000.
df_p.loc[df_p['predict_FuelRate[L/h]']<0, 'predict_FuelRate[L/h]'] = 0.
print(f"累計行駛 {df_p['VehicleSpeed[km/h]'].sum()/3.6/1000:.2f} 公里")
print(f"量測用油 {df_p['EngineFuelRate[L/h]'].sum()/3600:.2f} 公升")
print(f"量測能效 {(df_p['VehicleSpeed[km/h]'].sum()/3.6/1000)/(df_p['EngineFuelRate[L/h]'].sum()/3600):.2f} 公里/公升")
print(f"預測用油 {df_p['predict_FuelRate[L/h]'].sum()/3600:.2f} 公升")
print(f"預測能效 {(df_p['VehicleSpeed[km/h]'].sum()/3.6/1000)/(df_p['predict_FuelRate[L/h]'].sum()/3600):.2f} 公里/公升")

累計行駛 106.16 公里
量測用油 32.40 公升
量測能效 3.28 公里/公升
預測用油 30.23 公升
預測能效 3.51 公里/公升
